In [1]:
import os
import re

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity

# ── CONFIG ────────────────────────────────────────────────────────────────
DATA_DIR      = '.'   # folder containing your .csv files
EXPECTED_HRS  = 8760  # target hours per (cleaned) year
FILENAME_TAG  = 'Total Load'  # part of filename to filter on
# ─────────────────────────────────────────────────────────────────────────

def load_and_clean_profile(path):
    df = pd.read_csv(path)

    # 1) Parse the ISO datetime
    df['ts'] = pd.to_datetime(
        df['datetime'],
        format='%Y-%m-%d %H:%M:%S',
        errors='coerce'
    )
    df = df.dropna(subset=['ts'])

    # 2) Drop the duplicated hour at DST fall-back
    #    (keeps first occurrence, drops the extra)
    df = df.drop_duplicates(subset=['ts'], keep='first')

    # 3) Remove Feb 29 so non-leap→8760, leap→8784
    df = df[~((df['ts'].dt.month == 2) & (df['ts'].dt.day == 29))]

    # 4) Ensure load is numeric
    df['load'] = pd.to_numeric(df['load'], errors='coerce')
    df = df.dropna(subset=['load'])

    # 5) Final length check
    if len(df) != EXPECTED_HRS:
        print(f"⚠️  {os.path.basename(path)} has {len(df)} rows; expected {EXPECTED_HRS}. Skipping.")
        return None

    # 6) Return a clean, indexed Series sorted by timestamp
    return df.set_index('ts')['load'].sort_index()

# ── MAIN ─────────────────────────────────────────────────────────────────
all_profiles = {}
for fn in os.listdir(DATA_DIR):
    if fn.endswith('.csv') and FILENAME_TAG in fn:
        m = re.search(r'_(\d{4})01010000-', fn)
        if not m: 
            continue
        year = int(m.group(1))
        prof = load_and_clean_profile(os.path.join(DATA_DIR, fn))
        if prof is not None:
            all_profiles[year] = prof

if len(all_profiles) < 2:
    raise RuntimeError("Need at least two valid years to compare!")

# Align into a DataFrame: rows=years, cols=hour-offset 0…8759
profiles_df = pd.DataFrame(
    {yr: s.values for yr, s in all_profiles.items()}
).T

print("✅ Cleaned profiles shape (years × hours):", profiles_df.shape)

# ── NORMALISE (by annual total) ───────────────────────────────────────────
norm = profiles_df.div(profiles_df.sum(axis=1), axis=0)

# ── MEDOID SELECTION ──────────────────────────────────────────────────────
eu_dist  = euclidean_distances(norm)
cos_dist = 1 - cosine_similarity(norm)

eu_medoid  = norm.index[np.argmin(eu_dist.mean(axis=1))]
cos_medoid = norm.index[np.argmin(cos_dist.mean(axis=1))]

print(f"📊 Representative year (Euclidean): {eu_medoid}")
print(f"📊 Representative year (Cosine):    {cos_medoid}")

# ── EXPORT ────────────────────────────────────────────────────────────────
norm.loc[eu_medoid].to_csv("euclidean_representative_profile.csv", index=False)
norm.loc[cos_medoid].to_csv("cosine_representative_profile.csv",  index=False)

print("✅ Saved: euclidean_representative_profile.csv, cosine_representative_profile.csv")


✅ Cleaned profiles shape (years × hours): (10, 8760)
📊 Representative year (Euclidean): 2019
📊 Representative year (Cosine):    2019
✅ Saved: euclidean_representative_profile.csv, cosine_representative_profile.csv
